In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1E7XrGddwAIQ8Gsq9JOHY7Zs_EFxCwV__", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import math

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"\u2705 GPU available: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"   Memory: {total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("\u26a0\ufe0f No GPU detected. This notebook runs fine on CPU.")
    print("   Go to Runtime \u2192 Change runtime type \u2192 GPU (optional)")

print(f"\n\U0001f4e6 Python {sys.version.split()[0]}")
print(f"\U0001f525 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\U0001f3b2 Random seed set to {SEED}")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

%matplotlib inline

# \U0001f3d7\ufe0f Hybrid Attention + MoE

*Part 4 of the Vizuara series on Building Qwen3.5 from Scratch*
*Estimated time: ~90 minutes*

In Notebooks 1-3 we built up the full Gated DeltaNet layer -- an efficient $O(d^2)$ attention mechanism with error-correcting memory and learnable gating. But no single mechanism is perfect. Gated DeltaNet is fast but compresses information into a fixed-size state matrix. Standard softmax attention is precise but $O(N^2)$. What if we could get the best of both?

In this notebook, we will:
1. **Implement Gated Attention** -- a softmax attention variant with a learnable sigmoid gate
2. **Build the 3:1 hybrid pattern** -- 3 Gated DeltaNet layers followed by 1 Gated Attention layer, repeated
3. **Implement fine-grained Mixture-of-Experts (MoE)** -- 512 tiny expert networks with top-k routing and a shared expert
4. **Assemble the complete Qwen3.5 block** -- HybridAttn + MoE FFN with residual connections and RMSNorm
5. **Visualize expert routing** -- see which experts fire for which tokens and verify load balancing

# \U0001f916 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** -- it has already read this entire notebook and can help with concepts, code, and exercises.

**[\U0001f449 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/build-llm/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

---

## 1. \U0001f504 Recap: Gated DeltaNet from Notebook 3

Before we build the hybrid, let us bring back the key components we built in previous notebooks. We need:

1. **RMSNorm** -- the normalization layer used throughout Qwen3.5
2. **SwiGLU** -- the gated feedforward function (used in dense blocks)
3. **Gated DeltaNet** -- our efficient $O(d^2)$ attention mechanism

We will implement compact versions here so this notebook is self-contained.

In [ ]:
# ──────────────────────────────────────────────
# Core building blocks from Notebooks 1-3
# ──────────────────────────────────────────────


In [ ]:
#@title 🎧 Code Walkthrough: Rmsnorm Swiglu Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_rmsnorm_swiglu_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019).

    Simpler than LayerNorm -- no mean subtraction, no bias.
    Used throughout Qwen3.5.
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., d_model)
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    """SwiGLU feedforward: (SiLU(xW1) * xW3) W2

    The standard dense FFN in Qwen3.5 (used when NOT using MoE).
    """
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


print("\u2705 RMSNorm and SwiGLU defined.")

# Quick sanity check
norm = RMSNorm(64)
swiglu = SwiGLU(64, 128)
x = torch.randn(2, 10, 64)
print(f"   RMSNorm: {x.shape} -> {norm(x).shape}")
print(f"   SwiGLU:  {x.shape} -> {swiglu(x).shape}")

In [ ]:
#@title 🎧 Code Walkthrough: Gated Deltanet Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_gated_deltanet_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class GatedDeltaNet(nn.Module):
    """Compact Gated DeltaNet from Notebook 3.

    Gated linear attention with delta rule updates:
        S_t = alpha_t * S_{t-1} + beta_t * k_t (v_t - S_{t-1}^T k_t)^T

    For this teaching notebook we use the recurrent (loop) form.
    The real implementation uses chunkwise parallelism for speed.
    """
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        # Gate projections (per-head scalars -> expanded to head dim)
        self.alpha_proj = nn.Linear(d_model, n_heads, bias=True)  # decay gate
        self.beta_proj = nn.Linear(d_model, n_heads, bias=True)   # write gate

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        H, d = self.n_heads, self.d_head

        # Project Q, K, V
        q = self.q_proj(x).view(B, T, H, d)  # (B, T, H, d)
        k = self.k_proj(x).view(B, T, H, d)
        v = self.v_proj(x).view(B, T, H, d)

        # L2-normalize keys (critical for delta rule stability)
        k = F.normalize(k, p=2, dim=-1)

        # Compute gates
        alpha = torch.sigmoid(self.alpha_proj(x))  # (B, T, H) -- decay
        beta = torch.sigmoid(self.beta_proj(x))     # (B, T, H) -- write

        # Recurrent delta rule pass (per head)
        # State S: (B, H, d, d)
        S = torch.zeros(B, H, d, d, device=x.device, dtype=x.dtype)
        outputs = []

        for t in range(T):
            k_t = k[:, t]        # (B, H, d)
            v_t = v[:, t]        # (B, H, d)
            q_t = q[:, t]        # (B, H, d)
            a_t = alpha[:, t]    # (B, H)
            b_t = beta[:, t]     # (B, H)

            # Current prediction: S^T @ k_t  -> (B, H, d)
            # S is (B, H, d_k, d_v), k_t is (B, H, d_k) -> want (B, H, d_v)
            prediction = torch.einsum('bhkv,bhk->bhv', S, k_t)

            # Error
            error = v_t - prediction  # (B, H, d)

            # Gated delta update
            # S_new = alpha * S + beta * outer(k, error)
            a_t_expanded = a_t.unsqueeze(-1).unsqueeze(-1)   # (B, H, 1, 1)
            b_t_expanded = b_t.unsqueeze(-1).unsqueeze(-1)   # (B, H, 1, 1)
            correction = torch.einsum('bhk,bhv->bhkv', k_t, error)  # (B, H, d, d)
            S = a_t_expanded * S + b_t_expanded * correction

            # Output: q^T S = S^T q -> (B, H, d)
            out_t = torch.einsum('bhkv,bhk->bhv', S, q_t)
            outputs.append(out_t)

        # Stack and project
        output = torch.stack(outputs, dim=1)  # (B, T, H, d)
        output = output.reshape(B, T, D)
        return self.o_proj(output)


# Sanity check
gdn = GatedDeltaNet(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
y = gdn(x)
print(f"\u2705 GatedDeltaNet: {x.shape} -> {y.shape}")
print(f"   Params: {sum(p.numel() for p in gdn.parameters()):,}")

---

## 2. \U0001f50d Gated Attention: The Global Refresh

### The Trade-off

Gated DeltaNet is efficient -- $O(d^2)$ per step instead of $O(N \cdot d)$ for softmax. But it compresses everything into a fixed-size state matrix $S \in \mathbb{R}^{d \times d}$. This compression is inevitably **lossy**. Details get smeared, especially for rare tokens or long-distance dependencies that require exact retrieval.

Standard softmax attention is the opposite: it directly computes pairwise similarity between every query and every key. Nothing is compressed. Retrieval is precise. But the cost is $O(N^2)$ -- quadratic in sequence length.

What if we used **both**? The insight behind Qwen3.5 is:

> Use GatedDeltaNet for most layers (fast streaming), but periodically insert a softmax attention layer as a "global refresh" -- a chance for the model to look back at the full context with exact retrieval.

### Gated Attention Formula

Qwen3.5 does not use vanilla softmax attention for its global layers. It uses **Gated Attention**:

$$\text{GatedAttn}(Q, K, V) = \sigma(g) \odot \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

where:
- $Q, K, V$ are the standard query, key, value projections
- $g = W_g x + b_g$ is a learned gate (one scalar per feature)
- $\sigma(\cdot)$ is the sigmoid function, producing values in $(0, 1)$
- $\odot$ is element-wise multiplication

The sigmoid gate acts as a per-feature **volume knob**. The model learns which features of the attention output to trust and which to suppress. This provides fine-grained control that plain softmax lacks.

**Analogy:** Think of GatedDeltaNet as taking notes from a lecture (fast, approximate). Gated Attention is periodically flipping back through the textbook to verify your notes (slow, precise). The gate controls how much you trust what you just read.

In [ ]:
# ──────────────────────────────────────────────
# Visualize: Why we need both mechanisms
# ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: GatedDeltaNet -- constant memory, lossy compression
seq_lens = [64, 128, 256, 512, 1024, 2048]
gdn_cost = [d * 64 for d in seq_lens]  # O(d^2) per step, but state is fixed
softmax_cost = [n * 64 for n in seq_lens]  # O(N*d)

# Normalized to show relative cost
gdn_per_token = [64**2 for _ in seq_lens]   # Constant per-token cost
softmax_per_token = [n * 64 for n in seq_lens]  # Grows with N

axes[0].plot(seq_lens, gdn_per_token, 's-', color='#1565c0', linewidth=2.5,
             markersize=7, label='GatedDeltaNet (O(d\u00b2)/token)')
axes[0].plot(seq_lens, softmax_per_token, 'o-', color='#e53935', linewidth=2.5,
             markersize=7, label='Softmax Attn (O(N\u00b7d)/token)')
axes[0].set_xlabel('Sequence Length (N)', fontsize=12)
axes[0].set_ylabel('Cost per Token (ops)', fontsize=12)
axes[0].set_title('Computation Cost', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_yscale('log')

# Panel 2: Memory capacity
# GatedDeltaNet: fixed state d x d = 4096 "slots"
# Softmax: stores all N tokens (KV cache grows linearly)
gdn_memory = [64**2 for _ in seq_lens]
softmax_memory = [n * 64 for n in seq_lens]

axes[1].plot(seq_lens, gdn_memory, 's-', color='#1565c0', linewidth=2.5,
             markersize=7, label='GatedDeltaNet (d\u00b2 fixed)')
axes[1].plot(seq_lens, softmax_memory, 'o-', color='#e53935', linewidth=2.5,
             markersize=7, label='Softmax (N\u00b7d grows)')
axes[1].set_xlabel('Sequence Length (N)', fontsize=12)
axes[1].set_ylabel('Memory (parameters)', fontsize=12)
axes[1].set_title('Memory Usage', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

# Panel 3: The sweet spot -- hybrid
# 3:1 ratio: 3 cheap + 1 expensive = 3*d^2 + N*d
hybrid_cost = [3 * 64**2 + n * 64 for n in seq_lens]
pure_softmax_cost = [4 * n * 64 for n in seq_lens]
pure_gdn_cost = [4 * 64**2 for _ in seq_lens]

axes[2].plot(seq_lens, pure_gdn_cost, 's--', color='#1565c0', linewidth=2,
             markersize=6, alpha=0.7, label='4\u00d7 GatedDeltaNet')
axes[2].plot(seq_lens, pure_softmax_cost, 'o--', color='#e53935', linewidth=2,
             markersize=6, alpha=0.7, label='4\u00d7 Softmax')
axes[2].plot(seq_lens, hybrid_cost, 'D-', color='#2e7d32', linewidth=2.5,
             markersize=7, label='3:1 Hybrid (Qwen3.5)')
axes[2].set_xlabel('Sequence Length (N)', fontsize=12)
axes[2].set_ylabel('Cost per 4-layer Block', fontsize=12)
axes[2].set_title('The 3:1 Sweet Spot', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].set_yscale('log')

plt.suptitle('Why Hybrid? Combining GatedDeltaNet + Softmax Attention',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left:   GatedDeltaNet has constant per-token cost; softmax grows linearly.")
print("Center: GatedDeltaNet uses fixed memory; softmax KV cache grows with N.")
print("Right:  The 3:1 hybrid sits close to the GatedDeltaNet cost curve")
print("        while retaining softmax's precise retrieval every 4th layer.")

---

## 3. \U0001f527 Your Turn: TODO #1 -- Implement Gated Attention

Implement a PyTorch module `GatedAttention(d_model, n_heads)` that computes:

$$\text{GatedAttn}(Q, K, V) = \sigma(g) \odot \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Requirements:**
- Standard Q, K, V projections (no bias)
- A gate projection: `g = W_g @ x + b_g`, producing `(B, T, d_model)` values
- Sigmoid activation on the gate
- Multi-head attention with proper reshaping
- **Causal mask**: upper-triangular entries set to $-\infty$ before softmax
- Element-wise multiply gate with attention output
- Output projection

**Shapes reference:**
- Input `x`: `(B, T, d_model)`
- Q, K, V after projection and reshape: `(B, n_heads, T, d_head)`
- Attention scores: `(B, n_heads, T, T)`
- Gate `g` after sigmoid: `(B, T, d_model)`

In [ ]:
#@title 🎧 Before You Start: Todo1 Gated Attention Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_todo1_gated_attention_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO #1 ============
# Implement Gated Attention.
#
# GatedAttn(Q,K,V) = sigmoid(g) * softmax(QK^T / sqrt(d_k)) V
#
# The gate g = W_g @ x + b_g gives a per-feature "volume knob"
# that modulates the softmax attention output.
#
# IMPORTANT: Use a causal mask so position i can only attend to j <= i.
# ============ TODO #1 ============

class GatedAttention(nn.Module):
    """Gated softmax attention -- the 'global refresh' layer in Qwen3.5.

    Every 4th layer uses this instead of GatedDeltaNet, giving the model
    a chance to do exact full-context retrieval.
    """
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5  # 1/sqrt(d_k)

        # Q, K, V projections (no bias, following Qwen convention)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        # Output projection
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        # Gate projection (WITH bias -- the gate needs a learnable offset)
        self.gate_proj = nn.Linear(d_model, d_model, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, d_model)
        Returns:
            output: (B, T, d_model)
        """
        B, T, D = x.shape
        H, d = self.n_heads, self.d_head

        # Step 1: Project Q, K, V and reshape for multi-head
        # Shape: (B, T, D) -> (B, T, H, d) -> (B, H, T, d)
        q = ???  # YOUR CODE: project x, reshape, transpose
        k = ???  # YOUR CODE
        v = ???  # YOUR CODE

        # Step 2: Compute scaled dot-product attention scores
        # scores: (B, H, T, T)
        scores = ???  # YOUR CODE: Q @ K^T * scale

        # Step 3: Apply causal mask
        # Create upper triangular mask (True = positions to mask OUT)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = ???  # YOUR CODE: fill masked positions with -inf

        # Step 4: Softmax over the last dimension
        attn_weights = ???  # YOUR CODE

        # Step 5: Compute attention output
        # attn_output: (B, H, T, d)
        attn_output = ???  # YOUR CODE: weights @ V

        # Step 6: Reshape back to (B, T, D)
        attn_output = ???  # YOUR CODE: transpose and reshape

        # Step 7: Compute the sigmoid gate
        gate = ???  # YOUR CODE: sigmoid(gate_proj(x))

        # Step 8: Apply gate and output projection
        output = ???  # YOUR CODE: gate * attn_output, then o_proj

        return output

In [ ]:
#@title 🎧 Code Walkthrough: Todo1 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_todo1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    torch.manual_seed(42)
    d_model, n_heads = 64, 4
    ga = GatedAttention(d_model, n_heads)
    x = torch.randn(2, 10, d_model)

    output = ga(x)

    # Check 1: Output shape
    assert output.shape == (2, 10, d_model), \
        f"Expected shape (2, 10, {d_model}), got {output.shape}"
    print(f"\u2705 Check 1: Output shape correct: {output.shape}")

    # Check 2: Gate values are in (0, 1)
    with torch.no_grad():
        gate_values = torch.sigmoid(ga.gate_proj(x))
        assert gate_values.min() >= 0 and gate_values.max() <= 1, \
            "Gate values should be in [0, 1]"
        print(f"\u2705 Check 2: Gate values in (0, 1): min={gate_values.min():.4f}, max={gate_values.max():.4f}")

    # Check 3: When gate = 1 everywhere, output should match standard attention
    # Set gate bias to a large positive number so sigmoid -> 1
    with torch.no_grad():
        old_weight = ga.gate_proj.weight.clone()
        old_bias = ga.gate_proj.bias.clone()
        ga.gate_proj.weight.zero_()
        ga.gate_proj.bias.fill_(10.0)  # sigmoid(10) ~ 1.0

        output_gated = ga(x)

        # Compute standard attention manually
        q = ga.q_proj(x).view(2, 10, n_heads, d_model // n_heads).transpose(1, 2)
        k = ga.k_proj(x).view(2, 10, n_heads, d_model // n_heads).transpose(1, 2)
        v = ga.v_proj(x).view(2, 10, n_heads, d_model // n_heads).transpose(1, 2)
        scale = (d_model // n_heads) ** -0.5
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale
        causal_mask = torch.triu(torch.ones(10, 10, device=x.device, dtype=torch.bool), diagonal=1)
        scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_w = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_w, v)
        attn_out = attn_out.transpose(1, 2).reshape(2, 10, d_model)
        gate_approx = torch.sigmoid(ga.gate_proj(x))  # should be ~1.0
        expected = ga.o_proj(gate_approx * attn_out)

        diff = (output_gated - expected).abs().max().item()
        assert diff < 1e-5, f"When gate=1, should match standard attention. Diff={diff}"
        print(f"\u2705 Check 3: With gate\u22481, matches standard attention (max diff={diff:.2e})")

        # Restore
        ga.gate_proj.weight.copy_(old_weight)
        ga.gate_proj.bias.copy_(old_bias)

    # Check 4: Causal -- changing future tokens should not affect past outputs
    x2 = x.clone()
    x2[:, 5:, :] = torch.randn(2, 5, d_model)  # Change tokens 5-9
    output2 = ga(x2)
    # Outputs for tokens 0-4 should be identical
    causal_diff = (output[:, :5] - output2[:, :5]).abs().max().item()
    assert causal_diff < 1e-5, f"Causal violation! Diff={causal_diff}"
    print(f"\u2705 Check 4: Causality verified (future tokens don't affect past, diff={causal_diff:.2e})")

    print(f"\n\u2705 All checks passed! Your GatedAttention is correct.")
    print(f"   Parameters: {sum(p.numel() for p in ga.parameters()):,}")

except NameError:
    print("\u274c Replace the ??? placeholders with your code.")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"\u274c Error: {e}")

---
### \u270b Stop and Think
Before looking at the solution:
1. Why does the gate use **sigmoid** (range 0-1) instead of **tanh** (range -1 to 1)?
2. The gate is computed from the **input** $x$, not from the attention output. Why is this important?
3. If the gate learned to always output 1.0, what would GatedAttention reduce to?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class GatedAttention(nn.Module):
    """Gated softmax attention -- the 'global refresh' layer in Qwen3.5."""

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.gate_proj = nn.Linear(d_model, d_model, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        H, d = self.n_heads, self.d_head

        # Step 1: Project and reshape for multi-head
        q = self.q_proj(x).view(B, T, H, d).transpose(1, 2)  # (B, H, T, d)
        k = self.k_proj(x).view(B, T, H, d).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, d).transpose(1, 2)

        # Step 2: Scaled dot-product scores
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B, H, T, T)

        # Step 3: Causal mask
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Step 4: Softmax
        attn_weights = F.softmax(scores, dim=-1)  # (B, H, T, T)

        # Step 5: Weighted sum of values
        attn_output = torch.matmul(attn_weights, v)  # (B, H, T, d)

        # Step 6: Reshape back
        attn_output = attn_output.transpose(1, 2).reshape(B, T, D)  # (B, T, D)

        # Step 7: Sigmoid gate
        gate = torch.sigmoid(self.gate_proj(x))  # (B, T, D)

        # Step 8: Gate * attention output, then project
        output = self.o_proj(gate * attn_output)

        return output


# Verify
torch.manual_seed(42)
ga = GatedAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
y = ga(x)
print(f"\u2705 GatedAttention: {x.shape} -> {y.shape}")
print(f"   Parameters: {sum(p.numel() for p in ga.parameters()):,}")

---

## 4. \U0001f504 The 3:1 Hybrid Pattern

### Why 3:1?

Now we have two attention mechanisms:
- **GatedDeltaNet**: $O(d^2)$ per token, efficient but lossy
- **GatedAttention**: $O(N \cdot d)$ per token, expensive but precise

How should we mix them? Qwen3.5 uses a **3:1 pattern**: three GatedDeltaNet layers followed by one GatedAttention layer, repeated throughout the network.

For the 397B model with 60 layers:
- 45 GatedDeltaNet layers (75%)
- 15 GatedAttention layers (25%)

### Why not other ratios?

| Ratio | Pattern | Speed | Retrieval | Verdict |
|-------|---------|-------|-----------|---------|
| 4:0 | All GatedDeltaNet | Fastest | Poor (no exact retrieval) | Too lossy |
| 1:1 | Alternating | Moderate | Great | Too slow |
| 3:1 | 3 GDN + 1 GA | ~4x faster | Good (periodic refresh) | **Sweet spot** |
| 7:1 | 7 GDN + 1 GA | Very fast | Mediocre | Refresh too rare |

### Cost analysis for a 4-layer block

**Pure Transformer** (4 softmax layers):
$$\text{Cost} = 4 \times O(N \cdot d) = O(4Nd)$$

**3:1 Hybrid** (3 GatedDeltaNet + 1 GatedAttention):
$$\text{Cost} = 3 \times O(d^2) + 1 \times O(N \cdot d) = O(3d^2 + Nd)$$

When $N \gg d$ (long sequences), the hybrid is approximately **4x cheaper** than a pure Transformer. In practice, Qwen3.5 reports 8.6-19x faster decoding thanks to the reduced KV cache (only 15 layers need it instead of 60).

In [ ]:
# ──────────────────────────────────────────────
# Visualize the 3:1 hybrid pattern
# ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(18, 6))

n_layers = 24  # Show 24 layers (6 repeating units of 4)
colors = []
labels = []

for i in range(n_layers):
    if i % 4 == 3:  # Every 4th layer (0-indexed: 3, 7, 11, ...)
        colors.append('#e53935')  # Red for GatedAttention
        labels.append('GA')
    else:
        colors.append('#1565c0')  # Blue for GatedDeltaNet
        labels.append('GDN')

# Draw blocks
bar_width = 0.8
for i in range(n_layers):
    rect = mpatches.FancyBboxPatch(
        (i - bar_width/2, 0), bar_width, 1,
        boxstyle="round,pad=0.05",
        facecolor=colors[i], edgecolor='white', linewidth=2,
        alpha=0.9
    )
    ax.add_patch(rect)
    ax.text(i, 0.5, labels[i], ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    ax.text(i, -0.15, str(i), ha='center', va='top', fontsize=8, color='gray')

# Add group brackets
for group in range(n_layers // 4):
    start = group * 4
    ax.annotate('', xy=(start - 0.4, 1.15), xytext=(start + 3.4, 1.15),
                arrowprops=dict(arrowstyle='-', color='gray', lw=1.5))
    ax.text(start + 1.5, 1.25, f'Unit {group+1}', ha='center', fontsize=9, color='gray')

ax.set_xlim(-1, n_layers)
ax.set_ylim(-0.4, 1.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Qwen3.5 Hybrid Attention Pattern (3:1)\nBlue = GatedDeltaNet, Red = GatedAttention',
             fontsize=14, fontweight='bold', pad=20)

# Legend
gdn_patch = mpatches.Patch(color='#1565c0', label='GatedDeltaNet (O(d\u00b2), efficient)')
ga_patch = mpatches.Patch(color='#e53935', label='GatedAttention (O(N\u00b7d), precise)')
ax.legend(handles=[gdn_patch, ga_patch], loc='lower center', fontsize=11,
          ncol=2, bbox_to_anchor=(0.5, -0.15))

plt.tight_layout()
plt.show()

# Count
n_gdn = sum(1 for l in labels if l == 'GDN')
n_ga = sum(1 for l in labels if l == 'GA')
print(f"\nIn {n_layers} layers: {n_gdn} GatedDeltaNet + {n_ga} GatedAttention")
print(f"Ratio: {n_gdn}:{n_ga} = {n_gdn/n_ga:.0f}:1")
print(f"\nFor Qwen3.5 (60 layers): 45 GatedDeltaNet + 15 GatedAttention")

In [ ]:
# ──────────────────────────────────────────────
# Build the hybrid attention dispatcher
# ──────────────────────────────────────────────

class HybridAttention(nn.Module):
    """Dispatches to GatedDeltaNet or GatedAttention based on layer index.

    Pattern: layers 0,1,2 use GatedDeltaNet; layer 3 uses GatedAttention.
    Generalizes to any layer via (layer_idx % 4 == 3).
    """
    def __init__(self, d_model: int, n_heads: int, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.use_softmax = (layer_idx % 4 == 3)

        if self.use_softmax:
            self.attn = GatedAttention(d_model, n_heads)
        else:
            self.attn = GatedDeltaNet(d_model, n_heads)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.attn(x)

    @property
    def attn_type(self) -> str:
        return "GatedAttention" if self.use_softmax else "GatedDeltaNet"


# Test all 4 positions in a repeating unit
print("Hybrid attention dispatcher:")
print(f"{'Layer':>6} {'Type':>20} {'Mechanism':>15}")
print("-" * 45)

d_model, n_heads = 64, 4
x = torch.randn(2, 10, d_model)

for i in range(8):  # 2 full repeating units
    hybrid = HybridAttention(d_model, n_heads, layer_idx=i)
    y = hybrid(x)
    mechanism = "O(N\u00b7d) precise" if hybrid.use_softmax else "O(d\u00b2) efficient"
    print(f"{i:>6} {hybrid.attn_type:>20} {mechanism:>15}")

print(f"\n\u2705 Pattern verified: GDN, GDN, GDN, GA, GDN, GDN, GDN, GA, ...")

---

## 5. \U0001f9e0 Fine-Grained Mixture-of-Experts (MoE)

### The Problem with Standard FFN

In a standard Transformer, after attention, every token passes through the **same** feedforward network (FFN). If the FFN has `d_ff = 8192` hidden units, then **all 8192 neurons** activate for **every single token** -- whether it is a common word like "the" or a rare technical term like "eigendecomposition."

This is wasteful. Not every token needs the same amount of computation.

### The MoE Solution

**Mixture-of-Experts (MoE)** replaces the single large FFN with a collection of smaller "expert" networks. A **router** network examines each token and selects only a few experts to process it.

$$\text{MoE}(x) = g_s \cdot E_{\text{shared}}(x) + \sum_{i \in \text{TopK}(r(x), k)} g_i \cdot E_i(x)$$

where:
- $r(x) = W_r x$ is the **router**: a simple linear projection that scores each expert
- $\text{TopK}$ selects the $k$ highest-scoring experts
- $g_i = \text{softmax}_{i \in \text{TopK}}(r(x)_i)$ are the **gating weights** (softmax over selected scores only)
- $E_i(x)$ is expert $i$'s output (each expert is a small SwiGLU FFN)
- $E_{\text{shared}}(x)$ is a **shared expert** that processes every token
- $g_s$ is a learnable gate for the shared expert

### Qwen3.5 MoE Specs

| Component | Value |
|-----------|-------|
| Total experts | 512 |
| Active experts per token (top-k) | 10 |
| Expert hidden dim | 1,024 |
| Shared experts | 1 |
| Total parameters | 397B |
| Active parameters per token | ~17B (4.3%) |

### Why 512 Tiny Experts?

Previous MoE models (like Mixtral) used 8 large experts. Qwen3.5 uses 512 tiny ones. Why?

**Routing precision.** With 8 experts, each expert must handle ~12.5% of all token types. With 512, each expert can specialize in just ~0.2% of token types. This is like the difference between having 8 generalist doctors vs 512 specialists -- the specialists can be much more precise.

### The Shared Expert

One expert is special: the **shared expert**. It processes **every** token, regardless of what the router says. Why?

Some computations are universal -- syntax parsing, positional encoding, basic grammar. Instead of forcing every routed expert to redundantly learn these basics, the shared expert handles them once. The routed experts can then focus entirely on their specialized knowledge.

### Load Balancing

There is a danger with learned routing: the router might collapse to always selecting the same few experts (a "rich get richer" problem). To prevent this, we add a **load balancing loss**:

$$\mathcal{L}_{\text{balance}} = N_{\text{experts}} \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

where:
- $f_i$ = fraction of tokens routed to expert $i$
- $p_i$ = average router probability assigned to expert $i$

This loss is minimized when all experts receive roughly equal traffic.

In [ ]:
# ──────────────────────────────────────────────
# Visualize: Standard FFN vs MoE
# ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Standard FFN -- all parameters active
n_params = 100
active_dense = np.ones(n_params)

ax = axes[0]
colors_dense = ['#e53935'] * n_params  # All red (active)
ax.bar(range(n_params), active_dense, color=colors_dense, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Neuron Index', fontsize=12)
ax.set_ylabel('Active?', fontsize=12)
ax.set_title('Standard FFN: ALL Neurons Fire\n(100% utilization = wasteful)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Off', 'On'])
ax.text(50, 1.15, f'{n_params}/{n_params} active (100%)', ha='center', fontsize=12,
        fontweight='bold', color='#e53935')

# Panel 2: MoE -- sparse activation
n_experts = 100
top_k = 10
active_moe = np.zeros(n_experts)
# Randomly select top_k experts
np.random.seed(42)
selected = np.random.choice(n_experts, top_k, replace=False)
active_moe[selected] = 1

colors_moe = ['#1565c0' if active_moe[i] == 1 else '#e0e0e0' for i in range(n_experts)]
ax = axes[1]
ax.bar(range(n_experts), np.ones(n_experts), color=colors_moe, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Expert Index', fontsize=12)
ax.set_ylabel('Active?', fontsize=12)
ax.set_title('MoE: Only Top-K Experts Fire\n(10% utilization = efficient!)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Off', 'On'])
ax.text(50, 1.15, f'{top_k}/{n_experts} active ({100*top_k/n_experts:.0f}%)', ha='center',
        fontsize=12, fontweight='bold', color='#1565c0')

# Add shared expert annotation
ax.bar([0], [1], color='#2e7d32', edgecolor='white', linewidth=0.5)
ax.text(0, 1.05, 'S', ha='center', fontsize=9, fontweight='bold', color='#2e7d32')

plt.suptitle('Dense FFN vs Mixture-of-Experts', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left:  Standard FFN activates ALL neurons for every token. Wasteful!")
print("Right: MoE only activates the top-k experts (+ 1 shared expert).")
print("       Qwen3.5: 10/512 experts = only 4.3% of parameters active per token.")

---

## 6. \U0001f527 Your Turn: TODO #2 -- Implement Fine-Grained MoE

Implement `MoELayer(d_model, n_experts, top_k, expert_dim, has_shared)` with:

1. **Router**: Linear projection from `d_model` to `n_experts` scores
2. **Top-k selection**: Select the highest-scoring `top_k` experts per token
3. **Gating weights**: Softmax over the selected expert scores only
4. **Expert computation**: Each expert is a small SwiGLU FFN (`d_model -> expert_dim -> d_model`)
5. **Shared expert**: An additional SwiGLU FFN that always fires (with a learnable gate)
6. **Load balancing loss**: Stored in `self.aux_loss` after each forward pass

**Note for this demo**: We use `n_experts=32` and `expert_dim=128` to keep things fast on Colab. The real Qwen3.5 uses 512 experts with `expert_dim=1024`.

**Shapes reference:**
- Router scores: `(B, T, n_experts)`
- Top-k indices: `(B, T, top_k)`
- Top-k weights: `(B, T, top_k)` (softmax of selected scores)
- Each expert: `d_model -> expert_dim -> d_model`

In [ ]:
#@title 🎧 Before You Start: Todo2 Moe Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_19_todo2_moe_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO #2 ============
# Implement a Mixture-of-Experts layer.
#
# Key components:
#   1. Router: nn.Linear(d_model, n_experts) -- scores each expert per token
#   2. Top-k: torch.topk(scores, top_k) -- select best experts
#   3. Gating: F.softmax over selected scores only
#   4. Experts: nn.ModuleList of small SwiGLU FFNs
#   5. Shared expert: one SwiGLU FFN that always runs
#   6. Load balancing loss: stored in self.aux_loss
#
# The forward pass logic:
#   For each token:
#     1. Compute router scores
#     2. Select top-k experts
#     3. Compute gating weights (softmax over selected scores)
#     4. Run selected experts and combine with gating weights
#     5. Add shared expert output
#     6. Compute load balancing loss
# ============ TODO #2 ============

class Expert(nn.Module):
    """A single expert: a small SwiGLU FFN."""
    def __init__(self, d_model: int, d_expert: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_expert, bias=False)
        self.w3 = nn.Linear(d_model, d_expert, bias=False)
        self.w2 = nn.Linear(d_expert, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MoELayer(nn.Module):
    """Fine-grained Mixture-of-Experts layer.

    Uses many tiny experts with top-k routing and a shared expert.
    Demo config: 32 experts, top-4 routing, expert_dim=128.
    Qwen3.5 real config: 512 experts, top-10 routing, expert_dim=1024.
    """
    def __init__(self, d_model: int, n_experts: int = 32, top_k: int = 4,
                 expert_dim: int = 128, has_shared: bool = True):
        super().__init__()
        self.d_model = d_model
        self.n_experts = n_experts
        self.top_k = top_k
        self.has_shared = has_shared

        # Router: projects each token to expert scores
        self.router = ???  # YOUR CODE: nn.Linear(d_model, n_experts, bias=False)

        # Experts: a list of small SwiGLU FFNs
        self.experts = ???  # YOUR CODE: nn.ModuleList of Expert modules

        # Shared expert (always active)
        if has_shared:
            self.shared_expert = ???  # YOUR CODE: Expert(d_model, expert_dim)
            self.shared_gate = ???    # YOUR CODE: nn.Linear(d_model, 1, bias=True)

        # Auxiliary loss (set during forward)
        self.aux_loss = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, d_model)
        Returns:
            output: (B, T, d_model)
        """
        B, T, D = x.shape

        # Step 1: Compute router scores for each expert
        # router_scores: (B, T, n_experts)
        router_scores = ???  # YOUR CODE

        # Step 2: Select top-k experts per token
        # top_k_scores: (B, T, top_k) -- the scores of the selected experts
        # top_k_indices: (B, T, top_k) -- which experts were selected
        top_k_scores, top_k_indices = ???  # YOUR CODE: torch.topk(...)

        # Step 3: Compute gating weights (softmax over selected scores only)
        # gating_weights: (B, T, top_k) -- sums to 1 for each token
        gating_weights = ???  # YOUR CODE: F.softmax(top_k_scores, dim=-1)

        # Step 4: Compute weighted sum of expert outputs
        # For each token, run the selected experts and weight their outputs
        # output: (B, T, D)
        output = torch.zeros(B, T, D, device=x.device, dtype=x.dtype)

        # Flatten for efficient expert computation
        flat_x = x.reshape(B * T, D)                          # (B*T, D)
        flat_indices = top_k_indices.reshape(B * T, self.top_k)  # (B*T, top_k)
        flat_weights = gating_weights.reshape(B * T, self.top_k) # (B*T, top_k)
        flat_output = torch.zeros(B * T, D, device=x.device, dtype=x.dtype)

        # Loop over each expert and compute its contribution
        # (In production, this is done with scatter/gather ops for speed)
        for expert_idx in range(self.n_experts):
            # Find which (token, slot) pairs selected this expert
            # mask: (B*T, top_k) boolean -- True where this expert was selected
            mask = (flat_indices == expert_idx)  # (B*T, top_k)

            # Get the tokens that route to this expert
            token_mask = mask.any(dim=-1)  # (B*T,) -- True if any slot chose this expert

            if not token_mask.any():
                continue  # No tokens routed here, skip

            # Get the inputs for this expert
            expert_input = flat_x[token_mask]  # (n_tokens, D)

            # Run the expert
            expert_output = ???  # YOUR CODE: self.experts[expert_idx](expert_input)

            # Get the gating weights for this expert
            # For each token, sum the weights across slots that chose this expert
            expert_weights = (flat_weights * mask.float()).sum(dim=-1)  # (B*T,)
            expert_weights_selected = expert_weights[token_mask]  # (n_tokens,)

            # Weighted contribution
            flat_output[token_mask] += expert_weights_selected.unsqueeze(-1) * expert_output

        output = flat_output.reshape(B, T, D)

        # Step 5: Add shared expert
        if self.has_shared:
            shared_out = ???  # YOUR CODE: self.shared_expert(x)
            shared_g = ???    # YOUR CODE: sigmoid of self.shared_gate(x)
            output = output + shared_g * shared_out

        # Step 6: Compute load balancing loss
        # f_i = fraction of tokens routed to expert i
        # p_i = average router probability for expert i
        router_probs = F.softmax(router_scores, dim=-1)  # (B, T, n_experts)

        # Count how many tokens each expert handles
        # Using one-hot encoding of top-k indices
        one_hot = F.one_hot(top_k_indices, self.n_experts).float()  # (B, T, top_k, n_experts)
        tokens_per_expert = one_hot.sum(dim=2).sum(dim=(0, 1))  # (n_experts,)
        f_i = tokens_per_expert / (B * T)  # fraction

        # Average router probability per expert
        p_i = router_probs.mean(dim=(0, 1))  # (n_experts,)

        self.aux_loss = ???  # YOUR CODE: n_experts * (f_i * p_i).sum()

        return output

In [ ]:
#@title 🎧 Code Walkthrough: Todo2 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_20_todo2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    torch.manual_seed(42)
    d_model = 64
    n_experts = 32  # Small for demo (Qwen3.5 uses 512)
    top_k = 4       # Small for demo (Qwen3.5 uses 10)
    expert_dim = 128  # Small for demo (Qwen3.5 uses 1024)

    moe = MoELayer(d_model, n_experts=n_experts, top_k=top_k,
                   expert_dim=expert_dim, has_shared=True)
    x = torch.randn(2, 16, d_model)

    output = moe(x)

    # Check 1: Output shape
    assert output.shape == (2, 16, d_model), \
        f"Expected (2, 16, {d_model}), got {output.shape}"
    print(f"\u2705 Check 1: Output shape correct: {output.shape}")

    # Check 2: Only top_k experts fire per token
    with torch.no_grad():
        scores = moe.router(x)
        _, indices = torch.topk(scores, top_k, dim=-1)
        unique_experts = indices[0, 0].unique()
        assert len(unique_experts) == top_k, \
            f"Expected {top_k} experts per token, got {len(unique_experts)}"
        print(f"\u2705 Check 2: Exactly {top_k} experts fire per token")

    # Check 3: Gating weights sum to 1
    with torch.no_grad():
        top_scores, _ = torch.topk(scores, top_k, dim=-1)
        weights = F.softmax(top_scores, dim=-1)
        weight_sums = weights.sum(dim=-1)
        assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-5), \
            "Gating weights should sum to 1"
        print(f"\u2705 Check 3: Gating weights sum to 1 (mean={weight_sums.mean():.6f})")

    # Check 4: Load balancing loss is a positive scalar
    assert isinstance(moe.aux_loss, (float, torch.Tensor)), \
        f"aux_loss should be a scalar, got {type(moe.aux_loss)}"
    if isinstance(moe.aux_loss, torch.Tensor):
        loss_val = moe.aux_loss.item()
    else:
        loss_val = moe.aux_loss
    assert loss_val > 0, "Load balancing loss should be positive"
    print(f"\u2705 Check 4: Load balancing loss = {loss_val:.4f} (positive scalar)")

    # Check 5: Shared expert is always active
    if moe.has_shared:
        print(f"\u2705 Check 5: Shared expert present and active")

    total_params = sum(p.numel() for p in moe.parameters())
    expert_params = sum(p.numel() for e in moe.experts for p in e.parameters())
    print(f"\n\u2705 All checks passed!")
    print(f"   Total MoE params:   {total_params:,}")
    print(f"   Expert params:      {expert_params:,} ({n_experts} experts)")
    print(f"   Params per expert:  {expert_params // n_experts:,}")
    print(f"   Active per token:   {top_k} experts = {expert_params * top_k // n_experts:,} params ({100*top_k/n_experts:.1f}%)")

except NameError as e:
    print(f"\u274c Replace the ??? placeholders with your code. ({e})")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"\u274c Error: {e}")

---
### \u270b Stop and Think
Before looking at the solution:
1. Why do we use `softmax` over only the **selected** top-k scores, rather than over all expert scores?
2. What would happen if we did NOT have the shared expert? What types of knowledge would every routed expert need to redundantly learn?
3. If the load balancing loss drives all $f_i$ and $p_i$ to be uniform, what does that imply about expert specialization?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Todo2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_22_todo2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class Expert(nn.Module):
    """A single expert: a small SwiGLU FFN."""
    def __init__(self, d_model: int, d_expert: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_expert, bias=False)
        self.w3 = nn.Linear(d_model, d_expert, bias=False)
        self.w2 = nn.Linear(d_expert, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MoELayer(nn.Module):
    """Fine-grained Mixture-of-Experts layer."""

    def __init__(self, d_model: int, n_experts: int = 32, top_k: int = 4,
                 expert_dim: int = 128, has_shared: bool = True):
        super().__init__()
        self.d_model = d_model
        self.n_experts = n_experts
        self.top_k = top_k
        self.has_shared = has_shared

        # Router
        self.router = nn.Linear(d_model, n_experts, bias=False)

        # Experts
        self.experts = nn.ModuleList([
            Expert(d_model, expert_dim) for _ in range(n_experts)
        ])

        # Shared expert
        if has_shared:
            self.shared_expert = Expert(d_model, expert_dim)
            self.shared_gate = nn.Linear(d_model, 1, bias=True)

        self.aux_loss = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape

        # Step 1: Router scores
        router_scores = self.router(x)  # (B, T, n_experts)

        # Step 2: Top-k selection
        top_k_scores, top_k_indices = torch.topk(
            router_scores, self.top_k, dim=-1
        )  # both (B, T, top_k)

        # Step 3: Gating weights (softmax over selected only)
        gating_weights = F.softmax(top_k_scores, dim=-1)  # (B, T, top_k)

        # Step 4: Compute weighted expert outputs
        flat_x = x.reshape(B * T, D)
        flat_indices = top_k_indices.reshape(B * T, self.top_k)
        flat_weights = gating_weights.reshape(B * T, self.top_k)
        flat_output = torch.zeros(B * T, D, device=x.device, dtype=x.dtype)

        for expert_idx in range(self.n_experts):
            mask = (flat_indices == expert_idx)  # (B*T, top_k)
            token_mask = mask.any(dim=-1)        # (B*T,)

            if not token_mask.any():
                continue

            expert_input = flat_x[token_mask]
            expert_output = self.experts[expert_idx](expert_input)

            expert_weights = (flat_weights * mask.float()).sum(dim=-1)
            expert_weights_selected = expert_weights[token_mask]

            flat_output[token_mask] += expert_weights_selected.unsqueeze(-1) * expert_output

        output = flat_output.reshape(B, T, D)

        # Step 5: Shared expert
        if self.has_shared:
            shared_out = self.shared_expert(x)
            shared_g = torch.sigmoid(self.shared_gate(x))  # (B, T, 1)
            output = output + shared_g * shared_out

        # Step 6: Load balancing loss
        router_probs = F.softmax(router_scores, dim=-1)
        one_hot = F.one_hot(top_k_indices, self.n_experts).float()
        tokens_per_expert = one_hot.sum(dim=2).sum(dim=(0, 1))
        f_i = tokens_per_expert / (B * T)
        p_i = router_probs.mean(dim=(0, 1))
        self.aux_loss = self.n_experts * (f_i * p_i).sum()

        return output


# Verify
torch.manual_seed(42)
moe = MoELayer(d_model=64, n_experts=32, top_k=4, expert_dim=128, has_shared=True)
x = torch.randn(2, 16, 64)
y = moe(x)
print(f"\u2705 MoELayer: {x.shape} -> {y.shape}")
print(f"   Load balancing loss: {moe.aux_loss:.4f}" if isinstance(moe.aux_loss, float)
      else f"   Load balancing loss: {moe.aux_loss.item():.4f}")
print(f"   Total params: {sum(p.numel() for p in moe.parameters()):,}")

---

## 7. \U0001f4ca Visualizing Expert Routing

Let us see how the router distributes tokens across experts. We will create three visualizations:

1. **Expert activation heatmap**: which experts fire for which tokens
2. **Load distribution histogram**: how evenly are experts utilized?
3. **Routing weight distribution**: how confident is the router?

In [ ]:
# ──────────────────────────────────────────────
# Visualize expert routing patterns
# ──────────────────────────────────────────────

torch.manual_seed(42)
d_model = 64
n_experts = 32
top_k = 4

moe_viz = MoELayer(d_model, n_experts=n_experts, top_k=top_k,
                   expert_dim=128, has_shared=True)

# Simulate a short sequence
B, T = 1, 20  # Single batch, 20 tokens
x = torch.randn(B, T, d_model)

with torch.no_grad():
    _ = moe_viz(x)
    router_scores = moe_viz.router(x)  # (1, 20, 32)
    top_scores, top_indices = torch.topk(router_scores, top_k, dim=-1)  # (1, 20, 4)
    gating_w = F.softmax(top_scores, dim=-1)

# Build activation matrix: (T, n_experts) -- 1 if expert fires, 0 otherwise
activation = torch.zeros(T, n_experts)
for t in range(T):
    for slot in range(top_k):
        expert_id = top_indices[0, t, slot].item()
        activation[t, expert_id] = gating_w[0, t, slot].item()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Expert activation heatmap
im = axes[0].imshow(activation.numpy().T, aspect='auto', cmap='Blues',
                     interpolation='nearest')
axes[0].set_xlabel('Token Index', fontsize=12)
axes[0].set_ylabel('Expert Index', fontsize=12)
axes[0].set_title('Expert Activation Heatmap\n(brightness = routing weight)', fontsize=13,
                  fontweight='bold')
axes[0].set_xticks(range(0, T, 2))
plt.colorbar(im, ax=axes[0], label='Weight', shrink=0.8)

# Panel 2: Load distribution -- how many tokens each expert handles
tokens_per_expert = (activation > 0).float().sum(dim=0).numpy()  # Count of non-zero entries

colors_load = ['#1565c0' if t > 0 else '#e0e0e0' for t in tokens_per_expert]
axes[1].bar(range(n_experts), tokens_per_expert, color=colors_load, edgecolor='white')
axes[1].axhline(y=T * top_k / n_experts, color='#e53935', linestyle='--',
                linewidth=2, label=f'Ideal: {T * top_k / n_experts:.1f} tokens/expert')
axes[1].set_xlabel('Expert Index', fontsize=12)
axes[1].set_ylabel('Tokens Routed', fontsize=12)
axes[1].set_title('Expert Load Distribution\n(how many tokens each expert handles)',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

# Panel 3: Distribution of routing weights
all_weights = gating_w.reshape(-1).numpy()
axes[2].hist(all_weights, bins=30, color='#1565c0', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Routing Weight', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('Routing Weight Distribution\n(higher = router is more confident)',
                  fontsize=13, fontweight='bold')
mean_w = all_weights.mean()
axes[2].axvline(x=mean_w, color='#e53935', linestyle='--', linewidth=2,
                label=f'Mean = {mean_w:.3f}')
axes[2].axvline(x=1/top_k, color='gray', linestyle=':', linewidth=2,
                label=f'Uniform = {1/top_k:.3f}')
axes[2].legend(fontsize=10)

plt.suptitle(f'MoE Routing Analysis ({n_experts} experts, top-{top_k}, {T} tokens)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistics
n_active = (tokens_per_expert > 0).sum()
print(f"\nRouting statistics:")
print(f"  Experts used: {n_active}/{n_experts} ({100*n_active/n_experts:.0f}%)")
print(f"  Max tokens on one expert: {tokens_per_expert.max():.0f}")
print(f"  Min tokens on active expert: {tokens_per_expert[tokens_per_expert > 0].min():.0f}")
print(f"  Ideal (uniform): {T * top_k / n_experts:.1f} tokens per expert")
print(f"  Load balance loss: {moe_viz.aux_loss:.4f}" if isinstance(moe_viz.aux_loss, float)
      else f"  Load balance loss: {moe_viz.aux_loss.item():.4f}")

---

## 8. \U0001f527 Your Turn: TODO #3 -- Assemble the Complete Qwen3.5 Block

Now we put everything together. A single Qwen3.5 block consists of:

```
x -> RMSNorm -> HybridAttention -> + (residual) -> RMSNorm -> MoE FFN -> + (residual) -> output
```

Implement `Qwen35Block(d_model, n_heads, layer_idx, ...)` that:

1. Uses `HybridAttention` (which automatically dispatches to GatedDeltaNet or GatedAttention based on `layer_idx % 4`)
2. Uses `MoELayer` for the feedforward
3. Uses `RMSNorm` before each sub-layer (pre-norm architecture)
4. Uses residual connections around each sub-layer

Then stack 8 blocks and verify:
- The 3:1 pattern is correct (6 GDN + 2 GA)
- Output shape matches input shape
- Residual connections work (gradient flows)

In [ ]:
#@title 🎧 Before You Start: Todo3 Block Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_26_todo3_block_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO #3 ============
# Assemble the complete Qwen3.5 block.
#
# Architecture per block:
#   output = x + HybridAttention(RMSNorm(x))        -- attention + residual
#   output = output + MoELayer(RMSNorm(output))      -- FFN + residual
#
# Then stack 8 blocks into a mini-Qwen3.5 model.
# ============ TODO #3 ============

class Qwen35Block(nn.Module):
    """A single Qwen3.5 Transformer block.

    Pre-norm architecture:
        x -> RMSNorm -> HybridAttn -> + -> RMSNorm -> MoE FFN -> + -> out
                                      ^                          ^
                                residual                    residual
    """
    def __init__(self, d_model: int, n_heads: int, layer_idx: int,
                 n_experts: int = 32, top_k: int = 4, expert_dim: int = 128):
        super().__init__()
        self.layer_idx = layer_idx

        # Pre-attention norm
        self.attn_norm = ???  # YOUR CODE: RMSNorm(d_model)

        # Hybrid attention (dispatches based on layer_idx)
        self.attn = ???  # YOUR CODE: HybridAttention(d_model, n_heads, layer_idx)

        # Pre-FFN norm
        self.ffn_norm = ???  # YOUR CODE: RMSNorm(d_model)

        # MoE FFN
        self.ffn = ???  # YOUR CODE: MoELayer(d_model, n_experts, top_k, expert_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, d_model)
        Returns:
            output: (B, T, d_model)
        """
        # Sub-layer 1: Attention with pre-norm and residual
        h = ???  # YOUR CODE: x + attn(attn_norm(x))

        # Sub-layer 2: FFN with pre-norm and residual
        output = ???  # YOUR CODE: h + ffn(ffn_norm(h))

        return output

    @property
    def attn_type(self) -> str:
        return self.attn.attn_type


class MiniQwen35(nn.Module):
    """A mini Qwen3.5 model: stack of Qwen35Blocks.

    Uses the 3:1 hybrid pattern automatically via layer_idx.
    """
    def __init__(self, d_model: int = 64, n_heads: int = 4, n_layers: int = 8,
                 n_experts: int = 32, top_k: int = 4, expert_dim: int = 128):
        super().__init__()
        self.blocks = nn.ModuleList([
            ???  # YOUR CODE: Qwen35Block for each layer
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Pass through all blocks, then final norm."""
        for block in self.blocks:
            x = ???  # YOUR CODE: pass through block
        return self.final_norm(x)

    def get_aux_loss(self) -> torch.Tensor:
        """Sum load balancing losses from all MoE layers."""
        total = 0.0
        for block in self.blocks:
            loss = block.ffn.aux_loss
            if isinstance(loss, torch.Tensor):
                total = total + loss
            else:
                total = total + loss
        return total

In [ ]:
#@title 🎧 Code Walkthrough: Todo3 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_27_todo3_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    torch.manual_seed(42)
    d_model = 64
    n_heads = 4
    n_layers = 8

    model = MiniQwen35(d_model=d_model, n_heads=n_heads, n_layers=n_layers,
                       n_experts=32, top_k=4, expert_dim=128)

    # Check 1: 3:1 pattern
    print("Layer pattern:")
    gdn_count = 0
    ga_count = 0
    for i, block in enumerate(model.blocks):
        attn_type = block.attn_type
        marker = "\u2b50" if "GatedAttention" in attn_type else "  "
        print(f"  Layer {i}: {attn_type} {marker}")
        if "GatedAttention" in attn_type:
            ga_count += 1
        else:
            gdn_count += 1

    assert gdn_count == 6 and ga_count == 2, \
        f"Expected 6 GDN + 2 GA, got {gdn_count} GDN + {ga_count} GA"
    print(f"\n\u2705 Check 1: Pattern correct: {gdn_count} GatedDeltaNet + {ga_count} GatedAttention (3:1)")

    # Check 2: Forward pass
    x = torch.randn(2, 10, d_model)
    output = model(x)
    assert output.shape == x.shape, f"Expected {x.shape}, got {output.shape}"
    print(f"\u2705 Check 2: Output shape correct: {output.shape}")

    # Check 3: Residual connections (gradient flows)
    x_grad = torch.randn(2, 10, d_model, requires_grad=True)
    output_grad = model(x_grad)
    loss = output_grad.sum()
    loss.backward()
    assert x_grad.grad is not None, "Gradient should flow through residual connections"
    assert x_grad.grad.abs().sum() > 0, "Gradient should be non-zero"
    print(f"\u2705 Check 3: Gradient flows (grad norm = {x_grad.grad.norm():.4f})")

    # Check 4: Aux loss
    aux_loss = model.get_aux_loss()
    print(f"\u2705 Check 4: Total aux loss = {aux_loss:.4f}" if isinstance(aux_loss, float)
          else f"\u2705 Check 4: Total aux loss = {aux_loss.item():.4f}")

    # Model statistics
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n\u2705 All checks passed!")
    print(f"   Total model params: {total_params:,}")
    print(f"   Layers: {n_layers}")
    print(f"   Pattern: 3:1 (GatedDeltaNet : GatedAttention)")
    print(f"\n   Qwen3.5 397B would have:")
    print(f"   - 60 layers (45 GDN + 15 GA)")
    print(f"   - 512 experts per MoE layer")
    print(f"   - 397B total params, ~17B active per token")

except NameError as e:
    print(f"\u274c Replace the ??? placeholders with your code. ({e})")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"\u274c Error: {e}")

---
### \u270b Stop and Think
Before looking at the solution:
1. Why does Qwen3.5 use **pre-norm** (norm before attention/FFN) rather than **post-norm** (norm after)?
2. Without residual connections, what would happen to the gradient signal through 60 layers?
3. The MoE layer replaces the standard SwiGLU FFN. When would you choose a **dense** SwiGLU instead of MoE?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Todo3 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_29_todo3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class Qwen35Block(nn.Module):
    """A single Qwen3.5 Transformer block."""

    def __init__(self, d_model: int, n_heads: int, layer_idx: int,
                 n_experts: int = 32, top_k: int = 4, expert_dim: int = 128):
        super().__init__()
        self.layer_idx = layer_idx

        self.attn_norm = RMSNorm(d_model)
        self.attn = HybridAttention(d_model, n_heads, layer_idx)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = MoELayer(d_model, n_experts, top_k, expert_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm + attention + residual
        h = x + self.attn(self.attn_norm(x))

        # Pre-norm + MoE FFN + residual
        output = h + self.ffn(self.ffn_norm(h))

        return output

    @property
    def attn_type(self) -> str:
        return self.attn.attn_type


class MiniQwen35(nn.Module):
    """A mini Qwen3.5 model: stack of Qwen35Blocks."""

    def __init__(self, d_model: int = 64, n_heads: int = 4, n_layers: int = 8,
                 n_experts: int = 32, top_k: int = 4, expert_dim: int = 128):
        super().__init__()
        self.blocks = nn.ModuleList([
            Qwen35Block(d_model, n_heads, layer_idx=i,
                        n_experts=n_experts, top_k=top_k, expert_dim=expert_dim)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

    def get_aux_loss(self) -> torch.Tensor:
        total = 0.0
        for block in self.blocks:
            loss = block.ffn.aux_loss
            if isinstance(loss, torch.Tensor):
                total = total + loss
            else:
                total = total + loss
        return total


# Build and test
torch.manual_seed(42)
model = MiniQwen35(d_model=64, n_heads=4, n_layers=8,
                   n_experts=32, top_k=4, expert_dim=128)

x = torch.randn(2, 10, 64)
y = model(x)

print(f"\u2705 MiniQwen35: {x.shape} -> {y.shape}")
print(f"   Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"\n   Layer breakdown:")
for i, block in enumerate(model.blocks):
    print(f"   Layer {i}: {block.attn_type}")

---

## 9. \U0001f3a8 Visualizing the Complete Architecture

Let us create a comprehensive diagram showing how all the pieces fit together in a complete Qwen3.5 block.

In [ ]:
# ──────────────────────────────────────────────
# Architecture diagram: one Qwen3.5 block
# ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Complete Qwen3.5 Block Architecture', fontsize=16, fontweight='bold', pad=20)

# Helper to draw a rounded box
def draw_box(ax, x, y, w, h, text, color, fontsize=10, text_color='white'):
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.15",
        facecolor=color, edgecolor='white', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Input
draw_box(ax, 5.5, 0.2, 3, 0.6, 'Input: x  (B, T, d_model)', '#757575', fontsize=10)

# RMSNorm 1
draw_arrow(ax, 7, 0.8, 7, 1.3)
draw_box(ax, 5.5, 1.3, 3, 0.5, 'RMSNorm', '#ff9800', fontsize=10)

# Hybrid Attention
draw_arrow(ax, 7, 1.8, 7, 2.3)

# GatedDeltaNet box (left)
draw_box(ax, 2.5, 2.3, 3.5, 1.2, 'GatedDeltaNet\nO(d\u00b2) efficient\n(layers 0,1,2)', '#1565c0', fontsize=9)

# GatedAttention box (right)
draw_box(ax, 8, 2.3, 3.5, 1.2, 'GatedAttention\nO(N\u00b7d) precise\n(layer 3)', '#e53935', fontsize=9)

# "OR" connector
ax.text(7, 2.9, 'layer_idx % 4', ha='center', va='center', fontsize=9,
        color='#333', style='italic')

# Add + Residual
draw_arrow(ax, 7, 3.5, 7, 4.0)
draw_box(ax, 6.2, 4.0, 1.6, 0.5, 'Add', '#4caf50', fontsize=10)

# Residual arrow from input
ax.annotate('', xy=(6.2, 4.25), xytext=(1.5, 4.25),
            arrowprops=dict(arrowstyle='->', color='#4caf50', lw=1.5, ls='--'))
ax.annotate('', xy=(1.5, 4.25), xytext=(1.5, 0.5),
            arrowprops=dict(arrowstyle='-', color='#4caf50', lw=1.5, ls='--'))
ax.annotate('', xy=(1.5, 0.5), xytext=(5.5, 0.5),
            arrowprops=dict(arrowstyle='-', color='#4caf50', lw=1.5, ls='--'))
ax.text(1.0, 2.5, 'residual', ha='center', va='center', fontsize=8, color='#4caf50', rotation=90)

# RMSNorm 2
draw_arrow(ax, 7, 4.5, 7, 5.0)
draw_box(ax, 5.5, 5.0, 3, 0.5, 'RMSNorm', '#ff9800', fontsize=10)

# MoE FFN
draw_arrow(ax, 7, 5.5, 7, 6.0)
draw_box(ax, 4, 6.0, 6, 1.5, 'MoE FFN\nRouter \u2192 Top-K Selection \u2192 Expert Computation\n+ Shared Expert',
         '#7b1fa2', fontsize=9)

# Add + Residual 2
draw_arrow(ax, 7, 7.5, 7, 8.0)
draw_box(ax, 6.2, 8.0, 1.6, 0.5, 'Add', '#4caf50', fontsize=10)

# Residual arrow from after first add
ax.annotate('', xy=(6.2, 8.25), xytext=(12.5, 8.25),
            arrowprops=dict(arrowstyle='->', color='#4caf50', lw=1.5, ls='--'))
ax.annotate('', xy=(12.5, 8.25), xytext=(12.5, 4.25),
            arrowprops=dict(arrowstyle='-', color='#4caf50', lw=1.5, ls='--'))
ax.annotate('', xy=(12.5, 4.25), xytext=(7.8, 4.25),
            arrowprops=dict(arrowstyle='-', color='#4caf50', lw=1.5, ls='--'))
ax.text(13.0, 6.3, 'residual', ha='center', va='center', fontsize=8, color='#4caf50', rotation=90)

# Output
draw_arrow(ax, 7, 8.5, 7, 9.0)
draw_box(ax, 5.5, 9.0, 3, 0.6, 'Output: (B, T, d_model)', '#757575', fontsize=10)

plt.tight_layout()
plt.show()

print("Each Qwen3.5 block:")
print("  1. RMSNorm \u2192 HybridAttention (GDN or GA based on layer) \u2192 Residual Add")
print("  2. RMSNorm \u2192 MoE FFN (router + experts + shared) \u2192 Residual Add")
print("\nStack 60 of these blocks with the 3:1 pattern = full Qwen3.5!")

---

## 10. \U0001f9ea End-to-End: Forward Pass Through Mini-Qwen3.5

Let us run a full forward pass through our mini model and analyze what happens at each layer.

In [ ]:
# ──────────────────────────────────────────────
# Trace a forward pass through the mini model
# ──────────────────────────────────────────────

torch.manual_seed(42)
model = MiniQwen35(d_model=64, n_heads=4, n_layers=8,
                   n_experts=32, top_k=4, expert_dim=128)

B, T, D = 2, 20, 64
x = torch.randn(B, T, D)

# Hook to capture intermediate activations
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks
for i, block in enumerate(model.blocks):
    block.register_forward_hook(make_hook(f'block_{i}'))

# Forward pass
with torch.no_grad():
    output = model(x)

# Analyze layer-by-layer
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nLayer-by-layer activation statistics:")
print(f"{'Layer':<8} {'Type':<18} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 60)

# Input stats
print(f"{'Input':<8} {'---':<18} {x.mean():.4f} {x.std():.4f} {x.min():.4f} {x.max():.4f}")

norms = [x.norm().item()]
for i in range(8):
    act = activations[f'block_{i}']
    norms.append(act.norm().item())
    attn_type = model.blocks[i].attn_type
    short_type = 'GDN' if 'DeltaNet' in attn_type else 'GA'
    print(f"{i:<8} {short_type:<18} {act.mean():.4f} {act.std():.4f} "
          f"{act.min():.4f} {act.max():.4f}")

# Plot activation norms
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: Activation norms across layers
layer_names = ['Input'] + [f'L{i} ({("GA" if model.blocks[i].attn_type == "GatedAttention" else "GDN")})'
                            for i in range(8)]
colors_norm = ['#757575'] + ['#e53935' if model.blocks[i].attn_type == 'GatedAttention' else '#1565c0'
                               for i in range(8)]

axes[0].bar(range(len(norms)), norms, color=colors_norm, edgecolor='white', linewidth=1.5)
axes[0].set_xticks(range(len(norms)))
axes[0].set_xticklabels(layer_names, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Activation Norm', fontsize=12)
axes[0].set_title('Activation Norms Through the Network', fontsize=13, fontweight='bold')

# Panel 2: Expert utilization across all layers (combined)
total_expert_loads = np.zeros(32)
for i, block in enumerate(model.blocks):
    with torch.no_grad():
        if i == 0:
            block_input = model.blocks[0].attn_norm(x)
            block_input = model.blocks[0].attn(block_input) + x
        else:
            block_input = activations[f'block_{i-1}']
        router_scores = block.ffn.router(block.ffn_norm(block_input))
        _, top_idx = torch.topk(router_scores, 4, dim=-1)  # top_k=4
        for expert_id in range(32):
            total_expert_loads[expert_id] += (top_idx == expert_id).float().sum().item()

# Normalize
total_expert_loads = total_expert_loads / total_expert_loads.sum() * 100

axes[1].bar(range(32), total_expert_loads, color='#7b1fa2', edgecolor='white', alpha=0.8)
axes[1].axhline(y=100/32, color='#e53935', linestyle='--', linewidth=2,
                label=f'Ideal uniform: {100/32:.1f}%')
axes[1].set_xlabel('Expert Index', fontsize=12)
axes[1].set_ylabel('Token Share (%)', fontsize=12)
axes[1].set_title('Expert Utilization Across All Layers', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Mini-Qwen3.5 Forward Pass Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Aux loss
total_aux = model.get_aux_loss()
print(f"\nTotal load balancing loss: {total_aux:.4f}" if isinstance(total_aux, float)
      else f"\nTotal load balancing loss: {total_aux.item():.4f}")
print(f"This loss encourages the router to distribute tokens evenly across experts.")

---

## 11. \U0001f4b0 Cost Comparison: Why This Architecture Wins

Let us quantify exactly how much computation the hybrid MoE architecture saves compared to alternatives.

In [ ]:
# ──────────────────────────────────────────────
# Cost comparison: Qwen3.5 vs alternatives
# ──────────────────────────────────────────────

# Model configs (simplified for illustration)
d_model = 5120  # Qwen3.5 hidden size (approximate)
d_head = 128
n_heads = 40
n_layers = 60
d_ff = 8192     # Standard FFN hidden size
n_experts_real = 512
expert_dim_real = 1024
top_k_real = 10

seq_lens = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]

# Compute FLOPs per forward pass for different architectures
# 1. Pure Transformer (all softmax + dense FFN)
# 2. Pure Transformer (all softmax + MoE)
# 3. Qwen3.5 hybrid (3:1 GDN:GA + MoE)

pure_transformer_flops = []
pure_transformer_moe_flops = []
qwen35_flops = []

for N in seq_lens:
    # Attention FLOPs per layer (approximate: 2*N*d_model for Q@K^T + 2*N*d_model for attn@V)
    softmax_attn = 4 * N * d_model * N  # O(N^2 * d)
    gdn_attn = 4 * d_model * d_model    # O(d^2) -- constant in N

    # FFN FLOPs per layer
    dense_ffn = 2 * d_model * d_ff * 2  # 2 matmuls, each N * d_model * d_ff (per token)
    moe_ffn = top_k_real * 2 * d_model * expert_dim_real * 2  # Only top_k experts

    # 1. Pure Transformer (softmax + dense)
    pure_flops = n_layers * (softmax_attn + N * dense_ffn)
    pure_transformer_flops.append(pure_flops)

    # 2. Pure Transformer (softmax + MoE)
    pure_moe_flops = n_layers * (softmax_attn + N * moe_ffn)
    pure_transformer_moe_flops.append(pure_moe_flops)

    # 3. Qwen3.5 hybrid (3:1 + MoE)
    n_gdn = 45  # 3/4 of 60
    n_ga = 15   # 1/4 of 60
    hybrid_flops = n_gdn * (N * gdn_attn + N * moe_ffn) + n_ga * (softmax_attn + N * moe_ffn)
    qwen35_flops.append(hybrid_flops)

# Convert to TFLOPs
pure_transformer_tflops = [f / 1e12 for f in pure_transformer_flops]
pure_moe_tflops = [f / 1e12 for f in pure_transformer_moe_flops]
qwen35_tflops = [f / 1e12 for f in qwen35_flops]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Total FLOPs
axes[0].loglog(seq_lens, pure_transformer_tflops, 'o-', color='#e53935', linewidth=2.5,
               markersize=7, label='Pure Transformer\n(Softmax + Dense FFN)')
axes[0].loglog(seq_lens, pure_moe_tflops, 's-', color='#ff9800', linewidth=2.5,
               markersize=7, label='Transformer + MoE\n(Softmax + MoE FFN)')
axes[0].loglog(seq_lens, qwen35_tflops, 'D-', color='#1565c0', linewidth=2.5,
               markersize=7, label='Qwen3.5 Hybrid\n(3:1 GDN:GA + MoE)')
axes[0].set_xlabel('Sequence Length', fontsize=12)
axes[0].set_ylabel('FLOPs (TFLOPs)', fontsize=12)
axes[0].set_title('Computational Cost Comparison', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

# Panel 2: Speedup ratio
speedup_vs_pure = [p / q for p, q in zip(pure_transformer_flops, qwen35_flops)]
speedup_vs_moe = [p / q for p, q in zip(pure_transformer_moe_flops, qwen35_flops)]

axes[1].semilogx(seq_lens, speedup_vs_pure, 'o-', color='#e53935', linewidth=2.5,
                 markersize=7, label='vs Pure Transformer')
axes[1].semilogx(seq_lens, speedup_vs_moe, 's-', color='#ff9800', linewidth=2.5,
                 markersize=7, label='vs Transformer + MoE')
axes[1].set_xlabel('Sequence Length', fontsize=12)
axes[1].set_ylabel('Speedup (x)', fontsize=12)
axes[1].set_title('Qwen3.5 Speedup Ratio', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Why Hybrid Attention + MoE Wins at Scale',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"At N={seq_lens[-1]}:")
print(f"  vs Pure Transformer:     {speedup_vs_pure[-1]:.1f}x faster")
print(f"  vs Transformer + MoE:    {speedup_vs_moe[-1]:.1f}x faster")
print(f"\nThe hybrid pattern saves the most at long sequences,")
print(f"where the O(N\u00b2) cost of softmax attention dominates.")
print(f"\nQwen3.5 reported 8.6-19x faster decoding in practice")
print(f"(additional savings from reduced KV cache: only 15/60 layers need it).")

---

## 12. \U0001f3af Putting It All Together: The Full Picture

Let us summarize the complete Qwen3.5 architecture as we have built it across all four notebooks:

### The Journey So Far

| Notebook | Key Concept | What We Built |
|----------|-------------|---------------|
| NB1: Softmax to Linear | $O(N^2)$ bottleneck, linear attention reformulation | Linear attention recurrence |
| NB2: DeltaNet | Error-correcting memory via the delta rule | Delta rule update, memory benchmarks |
| NB3: Gated DeltaNet | Exponential gating for selective forgetting | Full GatedDeltaNet layer |
| **NB4: Hybrid + MoE** | **3:1 pattern, fine-grained MoE** | **Complete Qwen3.5 block** |

### The Complete Qwen3.5 Stack (60 layers)

```
[GDN+MoE] [GDN+MoE] [GDN+MoE] [GA+MoE]    \u2190 Unit 1  (layers 0-3)
[GDN+MoE] [GDN+MoE] [GDN+MoE] [GA+MoE]    \u2190 Unit 2  (layers 4-7)
[GDN+MoE] [GDN+MoE] [GDN+MoE] [GA+MoE]    \u2190 Unit 3  (layers 8-11)
        ...  (12 more units)  ...
[GDN+MoE] [GDN+MoE] [GDN+MoE] [GA+MoE]    \u2190 Unit 15 (layers 56-59)
```

**GDN** = GatedDeltaNet: $O(d^2)$ efficient attention with error-correcting memory + gating
**GA** = GatedAttention: $O(N \cdot d)$ precise softmax attention with sigmoid gate
**MoE** = 512 tiny experts, top-10 routing + 1 shared expert

### Key Numbers

| Metric | Value |
|--------|-------|
| Total parameters | 397B |
| Active parameters per token | ~17B (4.3%) |
| Attention layers (GDN) | 45 (75%) |
| Attention layers (GA) | 15 (25%) |
| Experts per MoE layer | 512 |
| Active experts per token | 10 + 1 shared |
| Reported decoding speedup | 8.6-19x vs pure Transformer |

---

## 13. \U0001f914 Reflection and Key Takeaways

### What We Learned

1. **Gated Attention** adds a learnable sigmoid gate to softmax attention, giving the model per-feature control over how much to trust the attention output. It serves as a "global refresh" layer that periodically provides exact full-context retrieval.

2. **The 3:1 hybrid pattern** is a carefully chosen balance between efficiency and quality. Three cheap GatedDeltaNet layers handle most of the streaming computation, while every fourth layer uses the expensive-but-precise GatedAttention for a "global refresh." This achieves ~4x theoretical speedup and 8.6-19x practical speedup during decoding.

3. **Fine-grained MoE** replaces the monolithic FFN with 512 tiny specialist experts. A router selects only 10 per token, activating just 4.3% of parameters. This allows the model to scale to 397B total parameters while keeping the per-token cost similar to a 17B dense model.

4. **The shared expert** handles universal computations (syntax, grammar, positional encoding) so that routed experts can focus entirely on specialized knowledge. This prevents redundancy and improves parameter efficiency.

5. **Load balancing** prevents router collapse by penalizing uneven expert utilization. Without it, the router would converge to always selecting the same few "popular" experts.

### Reflection Questions

1. **Why 3:1 and not 7:1 or 15:1?** The more GatedDeltaNet layers between GatedAttention refreshes, the more the compressed state drifts from the true attention output. How would you experimentally determine the optimal ratio?

2. **Why 512 tiny experts instead of 8 large ones?** Think about the routing precision argument. What are the downsides of having so many experts? (Hint: think about communication costs in distributed training.)

3. **The shared expert processes every token.** Could we have multiple shared experts? What would be the trade-off?

4. **The router is a simple linear projection.** Could we use a more complex router (e.g., a small MLP)? What are the pros and cons?

### Optional Challenges

1. **Vary the ratio:** Modify the `HybridAttention` class to support configurable ratios (e.g., 7:1, 1:1). Run the mini model with different ratios and compare output statistics.

2. **Expert analysis:** After a forward pass, visualize which experts specialize in which token positions. Do certain experts prefer early tokens? Late tokens? This gives insight into what the experts learn.

3. **Dense vs MoE:** Replace the MoE layer with a standard SwiGLU FFN of equivalent active parameter count. Compare the gradient norms. The MoE version should have more diverse gradients.

---

## 14. \U0001f680 What Comes Next

In **Notebook 5: Build & Run Mini-Qwen3.5 Locally**, we will:

1. Add token embeddings and a language modeling head to our mini model
2. Train it on a small dataset to verify it learns
3. Load official Qwen3.5 weights and run real inference
4. Quantize the model (INT4) to fit on consumer hardware
5. Run a full chat conversation locally

We have built every component of the Qwen3.5 architecture. Now it is time to see it in action.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_39_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final summary
print("""
\u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557
\u2551         HYBRID ATTENTION + MoE  CHEAT SHEET                \u2551
\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563
\u2551                                                               \u2551
\u2551  GATED ATTENTION (Global Refresh):                            \u2551
\u2551    GatedAttn(Q,K,V) = sigmoid(g) * softmax(QK^T/sqrt(d))V   \u2551
\u2551    g = W_g @ x + b_g  (learnable per-feature gate)           \u2551
\u2551    Cost: O(N * d) -- precise but expensive                    \u2551
\u2551                                                               \u2551
\u2551  3:1 HYBRID PATTERN:                                          \u2551
\u2551    [GDN] [GDN] [GDN] [GA] [GDN] [GDN] [GDN] [GA] ...       \u2551
\u2551    75% efficient + 25% precise = best of both                 \u2551
\u2551    ~4x theoretical speedup, 8.6-19x practical                 \u2551
\u2551                                                               \u2551
\u2551  MIXTURE-OF-EXPERTS (MoE):                                    \u2551
\u2551    MoE(x) = g_s*Shared(x) + sum_topk g_i*Expert_i(x)        \u2551
\u2551    512 tiny experts, top-10 routing, 1 shared expert          \u2551
\u2551    397B total params, ~17B active (4.3%)                      \u2551
\u2551                                                               \u2551
\u2551  COMPLETE BLOCK:                                              \u2551
\u2551    x -> RMSNorm -> HybridAttn -> +residual                   \u2551
\u2551      -> RMSNorm -> MoE FFN    -> +residual -> out            \u2551
\u2551                                                               \u2551
\u2551  KEY INSIGHT:                                                  \u2551
\u2551    Hybrid = speed (GDN) + precision (GA)                      \u2551
\u2551    MoE = scale (397B) + efficiency (17B active)               \u2551
\u2551    Together = a 397B model that runs like a 17B model         \u2551
\u2551                                                               \u2551
\u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d
""")
print("\u2705 Notebook complete! You now understand the full Qwen3.5 block architecture.")
print("\U0001f4ca Next up: Notebook 5 -- Build & Run Mini-Qwen3.5 Locally")